# Donut development: crop geometry diagnostic

Tests three predefined band pairs on the same 75 clean development receipts and 135 saved Qwen attack variants. It reuses each image's saved Qwen localization, runs only Donut readings, and keeps the original results unchanged. This is development tuning, not held-out evidence or direct Donut robustness testing.

**Use the existing Donut GPU runtime.** The original notebook's setup must already have created `/content/donut_band_study` and its Python environment. You can copy these code cells below the development-summary cell of that notebook. Do not run its freeze, evaluation, or adaptive cells yet. If the runtime was disconnected, rerun only the original setup/environment cells first.

Three candidates: smaller expansion (same first band, 1% extra margin); taller 18% minimum band; taller 24% minimum band with 4% context margins. The latter two retain a 3% expansion. All retain the existing 35%/45% height limits. No rule is chosen automatically, and agreement is not a correctness guarantee.


In [ ]:
from pathlib import Path
from google.colab import drive
import subprocess, os, json, zipfile

drive.mount('/content/drive')
ROOT = Path('/content/donut_band_study')
BASELINE = Path('/content/drive/MyDrive/CSCI 566 Project/test/csev_full_colab/full_study_results/donut_bands_hybrid_v1')
DIAG = BASELINE.parent / 'donut_geometry_development_v1'
PYTHON = ROOT / 'venv/bin/python'
assert PYTHON.is_file(), 'Run the original Donut notebook setup/environment cells first.'
assert (BASELINE/'development_summary.json').is_file(), f'Check BASELINE: {BASELINE}'
assert not (BASELINE/'frozen.json').exists(), 'Stop: the baseline has already been frozen.'
print('Baseline:', BASELINE)
print('New diagnostic results:', DIAG)
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.used,memory.total', '--format=csv'], check=True)


In [ ]:
DIAGNOSTIC_RUNNER = ROOT / "donut_geometry_diagnostic.py"
DIAGNOSTIC_SOURCE = '"""Development-only Donut geometry comparison; never selects a winning rule."""\nimport argparse, collections, csv, hashlib, json, os, sys, time\nfrom pathlib import Path\n\nVARIANTS = {\n    \'smaller_expansion\': (.12, .03, .01),\n    \'taller_18\': (.18, .03, .03),\n    \'taller_24\': (.24, .04, .03),\n}\n\ndef read(p): return json.loads(Path(p).read_text())\ndef sha(p):\n    h=hashlib.sha256()\n    with Path(p).open(\'rb\') as f:\n        for b in iter(lambda:f.read(8*1024*1024),b\'\'): h.update(b)\n    return h.hexdigest()\ndef write(p,x):\n    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)\n    t=p.with_suffix(\'.tmp\');t.write_text(json.dumps(x,indent=2,allow_nan=False));t.replace(p)\ndef boxes(tight,size,params):\n    if tight is None:return None,None\n    w,h=size;_,top,_,bottom=tight;minimum,margin,extra=params\n    height=max(round(minimum*h),bottom-top+2*round(margin*h))\n    if height<=0 or height>.35*h:return None,None\n    top=max(0,min(h-height,round((top+bottom-height)/2)))\n    first=(0,top,w,top+height);pad=round(extra*h)\n    second=(0,max(0,top-pad),w,min(h,top+height+pad))\n    if second[3]-second[1]>.45*h:return first,None\n    return first,second\n\ndef summarize(rows,out):\n    groups=collections.defaultdict(list)\n    for r in rows:\n        for ds in [r[\'dataset\'],\'ALL\']:\n            family=\'clean\' if r[\'family\']==\'clean\' else \'saved_transfer\'\n            for name,v in r[\'methods\'].items():groups[(ds,family,name)].append(v)\n    flat=[]\n    for (ds,fam,name),vv in sorted(groups.items()):\n        n=len(vv);c=sum(v[\'correct\'] for v in vv);a=sum(v[\'accepted\'] for v in vv)\n        flat.append(dict(dataset=ds,family=fam,method=name,n=n,correct=c,wrong=a-c,rejected=n-a,\n                         correct_rate=c/n,accepted_error_rate=(a-c)/n,coverage=a/n,\n                         targeted=sum(v[\'targeted\'] for v in vv)))\n    with (out/\'geometry_summary.csv\').open(\'w\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=list(flat[0]));w.writeheader();w.writerows(flat)\n    write(out/\'summary.json\',{\'completed_cases\':len(rows),\'expected_cases\':210,\'complete\':len(rows)==210,\n        \'note\':\'Development only. Saved attacks target Qwen, not Donut. No method automatically selected.\', \'groups\':flat})\n    for r in flat:\n        if r[\'dataset\']==\'ALL\':print(r,flush=True)\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--root\',required=True);p.add_argument(\'--baseline\',required=True);p.add_argument(\'--out\',required=True);a=p.parse_args()\n    root,baseline,out=map(Path,[a.root,a.baseline,a.out])\n    assert out.resolve()!=baseline.resolve() and not out.resolve().is_relative_to(baseline.resolve()),\'Use separate output folder.\'\n    assert not (baseline/\'frozen.json\').exists(),\'Baseline is frozen. Review before further development.\'\n    assert not list((baseline/\'evaluation\').glob(\'*.json\')),\'Reserved evaluation already started; do not silently tune.\'\n    roles=read(root/\'data_roles.json\');ids=set(roles[\'public_development\']+roles[\'collected_development\'])\n    assert len(ids)==75 and not ids & set(roles[\'collected_reserved_evaluation\'])\n    plan=read(root/\'plan.json\');docs={d[\'record_id\']:d for phase in [\'development\',\'evaluation\'] for d in plan[phase] if d[\'record_id\'] in ids}\n    tasks={}\n    for rid in sorted(ids):\n        d=docs[rid]\n        for item in [{\'family\':\'clean\',\'epsilon\':0,\'images\':d[\'images\']}]+d[\'attacks\']:\n            tasks[(rid,item[\'family\'],item[\'epsilon\'])]=item\n    files=sorted((baseline/\'development\').glob(\'*.json\'));old=[read(f) for f in files]\n    assert len(old)==len(tasks)==210\n    assert {(r[\'record_id\'],r[\'family\'],r[\'epsilon\']) for r in old}==set(tasks)\n    bc=read(baseline/\'contract.json\');sig=hashlib.sha256(json.dumps(bc,sort_keys=True).encode()).hexdigest()\n    for r in old:assert r[\'contract\']==sig\n    assert sha(root/\'source.tar\')==bc[\'source_archive\'] and sha(root/\'plan.json\')==bc[\'plan\']\n    assert sha(root/\'csev_receipts_v1.json\')==bc[\'config\'] and sha(root/\'data_roles.json\')==bc[\'roles\']\n    contract={\'script\':sha(__file__),\'variants\':VARIANTS,\'baseline_contract\':sig,\'baseline_records\':{f.name:sha(f) for f in files},\n              \'scope\':\'210 development cases only; saved per-image localization reused; no reserved evaluation\'}\n    # JSON round-trip normalizes tuples for restart comparisons.\n    contract=json.loads(json.dumps(contract));out.mkdir(parents=True,exist_ok=True)\n    if (out/\'contract.json\').exists():assert read(out/\'contract.json\')==contract,\'Diagnostic changed; use a new output folder.\'\n    else:write(out/\'contract.json\',contract)\n    sys.path.insert(0,str(root/\'source/src\'))\n    import torch, platform, importlib.metadata\n    from PIL import Image\n    from models.donut import DonutDocVQA\n    from normalization import normalize_receipt_total\n    env=read(baseline/\'environment.json\')\n    for k,v in env[\'versions\'].items():assert (platform.python_version() if k==\'python\' else importlib.metadata.version(k))==v,(k,v)\n    assert torch.cuda.is_available() and torch.cuda.get_device_name()==env[\'gpu\']\n    write(out/\'environment.json\',env)\n    os.environ[\'HF_HOME\']=str(root/\'hf_cache\');torch.manual_seed(566)\n    model=DonutDocVQA(device=torch.device(\'cuda\'),**bc[\'policy\'][\'reader\'])\n    prompt=read(root/\'csev_receipts_v1.json\')[\'legacy\'][\'prompts\'][\'receipt\']\n    rows=[]\n    try:\n        for i,(f,r) in enumerate(zip(files,old),1):\n            dest=out/\'cases\'/f.name\n            if dest.exists():rows.append(read(dest));print(f\'RESUME {i}/210\',flush=True);continue\n            item=tasks[(r[\'record_id\'],r[\'family\'],r[\'epsilon\'])][\'images\'][0]\n            imagepath=root/\'staging\'/str(item[\'path\']).lstrip(\'/\')\n            assert sha(imagepath)==item[\'sha256\']==r[\'input_sha256\']\n            image=Image.open(imagepath).convert(\'RGB\');o=r[\'observation\'];cache={}\n            for key,answer in [(\'box\',\'crop\'),(\'expanded_box\',\'wide\')]:\n                if o.get(key):cache[tuple(o[key])]={\'prediction\':o.get(answer),\'raw\':o.get(answer+\'_raw\'),\'reused\':True}\n            start=time.monotonic();methods=dict(r[\'methods\']);detail={}\n            for name,params in VARIANTS.items():\n                pair=boxes(o.get(\'tight_box\'),image.size,params);answers=[]\n                for box in pair:\n                    if box is None:answers.append(None);continue\n                    if box not in cache:\n                        raw=model.predict([image.crop(box)],prompt)\n                        cache[box]={\'prediction\':normalize_receipt_total(raw,r[\'dataset\']),\'raw\':raw,\'reused\':False}\n                    answers.append(cache[box][\'prediction\'])\n                pred=answers[0] if answers[0] is not None and answers[0]==answers[1] else None\n                methods[name]={\'prediction\':pred,\'accepted\':pred is not None,\'correct\':pred is not None and pred==r[\'clean_target\'],\n                               \'targeted\':pred is not None and pred==r[\'attack_target\']}\n                detail[name]={\'boxes\':pair,\'answers\':answers}\n            row={k:r[k] for k in [\'record_id\',\'dataset\',\'family\',\'epsilon\',\'clean_target\',\'attack_target\',\'input_sha256\']}\n            row.update(methods=methods,variants=detail,crops=[dict(box=k,**v) for k,v in cache.items()],elapsed_seconds=time.monotonic()-start)\n            write(dest,row);rows.append(row)\n            print(f\'{i}/210 {r["record_id"]} {r["family"]} eps={r["epsilon"]} \'+str({k:methods[k][\'prediction\'] for k in VARIANTS}),flush=True)\n            if i%10==0:summarize(rows,out)\n        summarize(rows,out)\n    finally:model.unload()\nif __name__==\'__main__\':main()\n'
DIAGNOSTIC_RUNNER.write_text(DIAGNOSTIC_SOURCE)
compile(DIAGNOSTIC_SOURCE, str(DIAGNOSTIC_RUNNER), "exec")
print("Diagnostic script ready.")


## Run development comparison
Outputs are saved per image in a separate Drive folder. Rerunning with identical code and inputs reuses completed cases. GPU progress prints after each image; summaries are updated every ten cases. Original baselines and localization are reused, reducing cost. No attack generation is run.


In [ ]:
DIAG.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env['MPLBACKEND'] = 'Agg'
with (DIAG/'diagnostic.log').open('a') as log:
    process = subprocess.Popen([str(PYTHON), '-u', str(DIAGNOSTIC_RUNNER), '--root', str(ROOT),
                                '--baseline', str(BASELINE), '--out', str(DIAG)],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    try:
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        status = process.wait()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=15)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
assert status == 0, f'Diagnostic failed (exit {status}); share diagnostic.log.'
print('Development comparison finished.')


## Review and export
Compare each dataset separately as well as the overall clean and saved-transfer groups. A lower accepted-error count achieved only by rejecting more correct answers is a trade-off, not an automatic improvement. Semantic consistency and the single-band control remain in the comparison. Do not modify settings after looking at reserved evaluation outcomes.


In [ ]:
import pandas as pd
summary = json.loads((DIAG/'summary.json').read_text())
print('Completed:', summary['completed_cases'], '/', summary['expected_cases'])
report = pd.read_csv(DIAG/'geometry_summary.csv')
keep = ['none', 'semantic_consistency', 'band_answer', 'verified_recovery',
        'smaller_expansion', 'taller_18', 'taller_24']
display(report[report.method.isin(keep)])


In [ ]:
from google.colab import files
archive = Path('/content/donut_geometry_development_v1_reports.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(DIAG.rglob('*')):
        if p.is_file() and p.suffix in {'.json', '.csv', '.log'}:
            z.write(p, p.relative_to(DIAG))
print('Share this ZIP; no source receipt images are included:', archive)
files.download(str(archive))
